In [17]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

In [18]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
[[255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
Using num_workers = 0


In [19]:
def evaluate_model(modelpath, modeltype, verbose=True, model_size=512):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=model_size, 
        dropout_rate=0.0, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [20]:
def evaluate_multiple_models(modeldict, modelsizes, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype, model_size=modelsizes[modelname])
        collected_results[modelname] = results
    return collected_results

In [21]:
basepath = os.path.join(os.path.expanduser("~"), "Library", "CloudStorage", "OneDrive-Nexus365", "BHF_Cardiac_Problem", "Results_Collection")

modeldict = {
    "BCE_Weighted": os.path.join(basepath, "Model_BCE_Weighted", "best_model.pth"), 
    "LR_Tuned": os.path.join(basepath, "Model_LR_Tuned", "best_model.pth")
}
modelsizes = {
    "BCE_Weighted": 512, 
    "LR_Tuned": 750
}

agg_results = evaluate_multiple_models(modeldict, modelsizes)

Loading from checkpoint, last run epoch was 14


Loading from checkpoint, last run epoch was 15


In [22]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name      Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
BCE_Weighted    0.764008           0.73913         0.792168    0.128025     0.128025       0.0916999       0.938651       0.934445    0.819971    0.798428
LR_Tuned        0.775324           0.774253        0.777827    0.0218274    0.0141758      0.0617093       0.941647       0.949255    0.836528    0.841687


In [23]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
------------  ---------  --------  --------  --------  --------
BCE_Weighted   0.756825  0.654843  0.747915  0.77918   0.881279
LR_Tuned       0.7621    0.663978  0.770279  0.798046  0.882217


In [24]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
------------  ----------------  ---------------  --------------  --------------  --------------
BCE_Weighted          0.70283          0.661157        0.731493        0.768274        0.831897
LR_Tuned              0.755405         0.660428        0.801418        0.812604        0.84141


In [25]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
------------  -------------  ------------  -----------  -----------  -----------
BCE_Weighted       0.819807      0.648649     0.765092       0.7904     0.936893
LR_Tuned           0.768913      0.667568     0.74147        0.784      0.927184


In [26]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
------------  ------------  -----------  ----------  ----------  ----------
BCE_Weighted      0.935698     0.916208    0.920775    0.934858    0.985716
LR_Tuned          0.936892     0.919575    0.93006     0.938635    0.983074
